In [19]:
# Step 1: Read the selected features from the output file generated in Part 2
selected_features_file = "output_ds.txt"

# Read the selected features from the file
with open(selected_features_file, "r") as file:
    selected_features = file.read().splitlines()

# Display the selected features
print("Number of selected features:", len(selected_features))
print("Selected features:", selected_features[:10])  # Display the first 10 selected features

from pyspark.sql import SparkSession
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, Normalizer, StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create a SparkSession
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()
# Suppress unnecessary logging
spark.sparkContext.setLogLevel("ERROR")

Number of selected features: 2000
Selected features: ['access', 'accessories', 'account', 'acid', 'acoustic', 'act', 'acted', 'acting', 'action', 'actions']


In [20]:
# Step 2: Load the dataset and preprocess the data
# Load the dataset
data_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json" 

# Load the dataset into a Spark DataFrame
dataframe = spark.read.json(data_path)

# Preprocess the data
# Tokenization
tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern="\s+|[,.;]", minTokenLength=2)
dataframe = tokenizer.transform(dataframe)

# Stopword removal
stop_words_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
dataframe = stop_words_remover.transform(dataframe)

# TF-IDF calculation
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[hashingTF, idf])
pipeline_model = pipeline.fit(dataframe)
dataframe = pipeline_model.transform(dataframe)

# Display the preprocessed dataframe schema and the first few rows
dataframe.printSchema()
dataframe.show(5)

# Step 3: Split the dataset into training, validation, and test sets
# Define the proportions for splitting the dataset
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Split the dataset into training, validation, and test sets
train_data, val_data, test_data = dataframe.randomSplit([train_ratio, val_ratio, test_ratio], seed=42)

# Cache the datasets for better performance
train_data.cache()
val_data.cache()
test_data.cache()

# Print the sizes of the datasets
print("Training set size:", train_data.count())
print("Validation set size:", val_data.count())
print("Test set size:", test_data.count())


root
 |-- asin: string (nullable = true)
 |-- category: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- overall: double (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)

+----------+--------------------+-------+-------+--------------------+-----------+--------------+--------------------+-------------------+--------------+--------------------+--------------------+--------------------+--------------------+
|      asin|            category|helpful|ove

Training set size: 63165


Validation set size: 7775


Test set size: 7889


In [21]:
# Step 4: Apply vector length normalization using Normalizer with L2 norm
# Create a Normalizer transformer
normalizer = Normalizer(inputCol="features", outputCol="normalized_features", p=2.0)

# Apply normalization to training, validation, and test data
train_data = normalizer.transform(train_data)
val_data = normalizer.transform(val_data)
test_data = normalizer.transform(test_data)

# Display the first few rows of the normalized features in the training set
train_data.select("normalized_features").show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|normalized_features                                                                                                                                                                                                                                                                                                                  

In [22]:
# Step 5: Prepare the SVM classifier within a pipeline
# Create a StringIndexer transformer
label_indexer = StringIndexer(inputCol="category", outputCol="label")

# Update the pipeline to include the StringIndexer
pipeline = Pipeline(stages=[label_indexer])

# Fit the pipeline to the training data
pipeline_model = pipeline.fit(train_data)
train_data_transformed = pipeline_model.transform(train_data)

In [23]:
# Step 6: Set up the OneVsRest SVM model
svm = LinearSVC(maxIter=10, regParam=0.1)
ovr = OneVsRest(classifier=svm)


In [24]:
# Step 7: Set up the parameter grid for grid search
paramGrid = ParamGridBuilder() \
    .addGrid(svm.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(svm.maxIter, [10, 50]) \
    .build()

In [25]:
# Step 8: Set up cross-validation
evaluator = MulticlassClassificationEvaluator(metricName="f1")

crossval = CrossValidator(estimator=ovr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)  # Use 3-fold cross-validation

In [26]:
# Step 9: Train the model using cross-validation
cvModel = crossval.fit(train_data_transformed)

In [28]:
# Step 10: Evaluate the model on the test set
test_data_transformed = pipeline_model.transform(test_data)
predictions = cvModel.transform(test_data_transformed)

f1_score = evaluator.evaluate(predictions)
print("Test set F1 score:", f1_score)

Test set F1 score: 0.5481380819703785


In [29]:
# Step 11: Save the coefficients of the best model
best_model = cvModel.bestModel
linear_svc_models = best_model.models

coefficients_list = []
for idx, model in enumerate(linear_svc_models):
    coefficients_list.append((f"Coefficients for class {idx}", model.coefficients))

# Display the learned SVM model coefficients
for class_idx, coefficients in coefficients_list:
    print(class_idx, ":", coefficients)

# Define the file path to save the coefficients
output_file_path = "svm_model_coefficients.txt"

# Open the file in write mode
with open(output_file_path, "w") as output_file:
    # Write the learned SVM model coefficients to the file
    for class_idx, coefficients in coefficients_list:
        output_file.write(f"{class_idx}: {coefficients}\n")

print("SVM model coefficients have been written to:", output_file_path)

Coefficients for class 0 : [0.012077961560628538,-0.08002917244345414,-0.037389675218169896,-0.1311962798223243,-0.025778004557081496,-0.014050681638926497,-0.0065418242942091205,0.04636467167300213,0.025824913635334845,-0.02744532258404102,-0.021218565269061268,0.022518898121339292,-0.0009103556256560597,-0.00013480663629813314,0.032052479475285854,0.0016197333247128336,-0.024699837920094888,-0.0018557678171117593,-0.020125918565474514,-0.03354819636347413,-0.013539959541839189,-0.007239924711910164,-0.032782581080568224,0.015158223792891449,-0.00018650580138172837,-0.019058855568273893,-0.00531805639586363,0.0195060347057614,0.04792960123548873,-0.00989658558368128,-0.015597554749281687,0.02712820952738829,-0.022391738145320958,-0.0009843620348822038,-0.018166145572658696,0.0047311697584836625,0.0044950987365403785,-0.0158945572694994,-0.00423009416245746,0.006612739007956238,0.02698703122969265,-0.0030277186341411953,-0.015268117655871256,-0.036209906495935355,0.0018242480988527158,

In [30]:
spark.stop()